# Module 4: Interactive Visualization with Plotly


## Section 2: Advanced Interactive Visualizations

### Objective

- Create complex multi-trace visualizations combining different chart types
- Master faceted plots for multi-dimensional comparisons
- Implement custom buttons and dropdown menus for user control
- Create synchronized multi-plot dashboards
- Use animations to show changes over time effectively
- Build 3D visualizations for complex spatial data
- Apply advanced styling and theming for brand consistency

### Main Contents with Examples

**Multi-Trace Figures - Combining Different Visualizations**

**The Power of Layered Information:**

Real business analysis often requires showing different data types on the same chart:

- Actual values + targets + forecasts
- Multiple metrics on same timeline
- Data + statistical summaries
- Comparisons with different chart types

**Example: Sales with Target Lines and Average:**


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Generate monthly sales data
np.random.seed(42)
months = pd.date_range('2024-01-01', '2024-12-31', freq='M')
actual_sales = np.random.normal(100000, 15000, 12)
target_sales = np.full(12, 110000)
forecast = actual_sales * 1.1  # Next year forecast

# Create figure with multiple traces
fig = go.Figure()

# Trace 1: Actual sales (bar chart)
fig.add_trace(go.Bar(
    x=months,
    y=actual_sales,
    name='Actual Sales',
    marker_color='lightblue',
    hovertemplate='<b>Actual</b><br>%{x|%b %Y}<br>$%{y:,.0f}<extra></extra>'
))

# Trace 2: Target line
fig.add_trace(go.Scatter(
    x=months,
    y=target_sales,
    name='Target',
    line=dict(color='red', width=3, dash='dash'),
    hovertemplate='<b>Target</b><br>$%{y:,.0f}<extra></extra>'
))

# Trace 3: Forecast line
fig.add_trace(go.Scatter(
    x=months,
    y=forecast,
    name='2025 Forecast',
    line=dict(color='green', width=2, dash='dot'),
    hovertemplate='<b>Forecast</b><br>%{x|%b %Y}<br>$%{y:,.0f}<extra></extra>'
))

# Trace 4: Average line
avg_sales = np.mean(actual_sales)
fig.add_hline(y=avg_sales, 
              line_dash='dash', 
              line_color='gray',
              annotation_text=f'Average: ${avg_sales:,.0f}',
              annotation_position='right')

# Update layout
fig.update_layout(
    title='2024 Sales Performance: Actual vs Target with 2025 Forecast',
    xaxis_title='Month',
    yaxis_title='Sales ($)',
    yaxis_tickformat='$,.0f',
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    height=600
)

fig.show()


**Output:**

*[Interactive Plotly chart: "2024 Sales Performance: Actual vs Target with 2025 Forecast"]*

**Understanding Multi-Trace Construction:**

**Why Use Graph Objects Here:**

- Mixing different chart types (bar + line)
- Need precise control over each trace
- Adding reference lines (hline)
- Complex hover templates

**Key Techniques:**

- `add_trace()`: Adds new data series
- `add_hline()` / `add_vline()`: Reference lines
- Each trace has independent styling
- `hovermode='x unified'`: Compare all traces at once

**Business Value:**

- Shows performance (actual)
- Shows expectations (target)
- Shows future (forecast)
- Shows context (average)
- All in ONE interactive chart

**Faceted Plots - Small Multiples for Comparison**

**The Small Multiples Principle:**

When comparing many groups, sometimes separate plots work better than overlaying:
- Each group gets full space
- Patterns within groups clearer
- Easy to spot outliers per group
- Still comparable (same scales)

**Creating Faceted Plots with Plotly Express:**


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Generate sales data for multiple products and regions
np.random.seed(42)
products = ['Product A', 'Product B', 'Product C', 'Product D']
regions = ['North', 'South', 'East', 'West']
months = pd.date_range('2024-01-01', '2024-12-31', freq='M')

data_list = []
for product in products:
    for region in regions:
        for month in months:
            # Different products have different patterns
            base = {'Product A': 50000, 'Product B': 35000, 
                    'Product C': 45000, 'Product D': 30000}[product]
            
            # Regional variation
            regional_mult = {'North': 1.1, 'South': 0.9, 
                           'East': 1.2, 'West': 1.0}[region]
            
            # Trend and noise
            month_num = month.month
            sales = base * regional_mult * (1 + 0.1 * np.sin(month_num/6 * np.pi)) + np.random.normal(0, 3000)
            
            data_list.append({
                'date': month,
                'product': product,
                'region': region,
                'sales': max(sales, 10000)
            })

df_sales = pd.DataFrame(data_list)

# Create faceted line chart
fig = px.line(df_sales,
              x='date',
              y='sales',
              color='region',
              facet_col='product',
              facet_col_wrap=2,  # 2 columns, multiple rows
              title='Sales Performance by Product and Region - 2024',
              labels={'sales': 'Monthly Sales ($)', 'date': 'Month'},
              height=800)

# Update all subplot titles
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))

# Format y-axes
fig.update_yaxes(tickformat='$,.0f')

# Improve facet spacing
fig.update_layout(
    showlegend=True,
    legend=dict(
        title='Region',
        orientation='v',
        yanchor='top',
        y=1,
        xanchor='left',
        x=1.02
    )
)

fig.show()


**Output:**

*[Interactive line chart: "Sales Performance by Product and Region - 2024", x: date, y: sales]*

**Faceting Parameters:**

**facet_col**: Creates columns of subplots

- `facet_col='product'`: One column per product

**facet_row**: Creates rows of subplots

- `facet_row='region'`: One row per region

**facet_col_wrap**: Wraps columns to multiple rows

- `facet_col_wrap=2`: Maximum 2 columns, then wrap to next row
- Useful when you have many categories

**When to Use Facets:**

- Comparing 4+ groups
- Each group has its own pattern
- Want to avoid cluttered single plot
- Groups should be compared on same scale

**Dropdown Menus and Buttons - User-Controlled Interactivity**

**Beyond Hover and Zoom:**

Sometimes users need to change WHAT data is displayed, not just HOW it's viewed:

- Switch between metrics (Revenue vs Profit vs Units)
- Change time aggregation (Daily vs Monthly vs Quarterly)
- Filter by category
- Toggle between chart types

**Creating Dropdown Menus:**


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Generate data
np.random.seed(42)
months = pd.date_range('2024-01-01', '2024-12-31', freq='M')
revenue = np.random.normal(100000, 15000, 12)
profit = revenue * np.random.uniform(0.15, 0.30, 12)
units = revenue / np.random.uniform(45, 55, 12)

# Create figure with multiple traces (initially all visible)
fig = go.Figure()

# Revenue trace
fig.add_trace(go.Scatter(
    x=months,
    y=revenue,
    name='Revenue',
    line=dict(color='blue', width=3),
    visible=True
))

# Profit trace
fig.add_trace(go.Scatter(
    x=months,
    y=profit,
    name='Profit',
    line=dict(color='green', width=3),
    visible=False  # Initially hidden
))

# Units trace
fig.add_trace(go.Scatter(
    x=months,
    y=units,
    name='Units Sold',
    line=dict(color='orange', width=3),
    visible=False  # Initially hidden
))

# Create dropdown menu
fig.update_layout(
    updatemenus=[
        dict(
            buttons=list([
                dict(
                    args=[{'visible': [True, False, False]},
                          {'yaxis': {'title': 'Revenue ($)', 'tickformat': '$,.0f'}}],
                    label='Revenue',
                    method='update'
                ),
                dict(
                    args=[{'visible': [False, True, False]},
                          {'yaxis': {'title': 'Profit ($)', 'tickformat': '$,.0f'}}],
                    label='Profit',
                    method='update'
                ),
                dict(
                    args=[{'visible': [False, False, True]},
                          {'yaxis': {'title': 'Units Sold', 'tickformat': ',.0f'}}],
                    label='Units',
                    method='update'
                ),
            ]),
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.11,
            xanchor='left',
            y=1.15,
            yanchor='top'
        ),
    ],
    title='Business Performance Dashboard - Select Metric',
    xaxis_title='Month',
    yaxis_title='Revenue ($)',
    yaxis_tickformat='$,.0f',
    height=600
)

fig.show()


**Output:**

*[Interactive Plotly chart: "Business Performance Dashboard - Select Metric"]*

**Understanding Update Menus:**

**Structure:**


In [ ]:
updatemenus=[
    dict(
        buttons=[...],      # List of button definitions
        direction='down',   # Dropdown direction
        showactive=True,    # Highlight selected button
        x=0.11,            # Position (0-1 scale)
        y=1.15             # Position (0-1 scale)
    )
]


**Button Definition:**


In [ ]:
dict(
    args=[{trace updates}, {layout updates}],
    label='Button Text',
    method='update'  # 'update' changes both traces and layout
)


**args Structure:**

- First dict: Trace properties (`{'visible': [True, False, False]}`)
- Second dict: Layout properties (`{'yaxis': {'title': 'New Title'}}`)

**Business Application:**

Executive dashboard where users select:

- What to view (Revenue, Profit, Costs)
- Appropriate Y-axis automatically updates
- Formatting changes based on metric
- One clean interface, multiple views

**Animation - Showing Change Over Time**

**When Animation Enhances Understanding:**

Animation is powerful when:

- Showing temporal evolution (year-over-year growth)
- Demonstrating process flows
- Revealing how relationships change
- Engaging presentation audiences

**Creating Animated Scatter Plot:**


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Generate data: Company metrics over 5 years
np.random.seed(42)
years = range(2020, 2025)
companies = ['TechCorp', 'InnovateLtd', 'DataSystems', 'CloudNet', 
             'AIVentures', 'DevOps Inc', 'CyberGuard']

data_list = []
for year in years:
    for company in companies:
        # Each company grows differently
        year_factor = (year - 2020) + 1
        
        revenue = np.random.uniform(10, 100) * year_factor
        profit_margin = np.random.uniform(0.05, 0.30)
        employees = int(revenue * np.random.uniform(5, 15))
        
        data_list.append({
            'year': year,
            'company': company,
            'revenue': revenue,
            'profit_margin': profit_margin,
            'employees': employees
        })

df_companies = pd.DataFrame(data_list)

# Create animated scatter plot
fig = px.scatter(df_companies,
                 x='revenue',
                 y='profit_margin',
                 animation_frame='year',
                 animation_group='company',
                 size='employees',
                 color='company',
                 hover_name='company',
                 size_max=60,
                 range_x=[0, 500],
                 range_y=[0, 0.35],
                 title='Company Growth Animation: Revenue vs Profitability (2020-2024)',
                 labels={
                     'revenue': 'Revenue ($M)',
                     'profit_margin': 'Profit Margin',
                     'year': 'Year'
                 })

# Format hover
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>' +
                  'Revenue: $%{x:.1f}M<br>' +
                  'Profit Margin: %{y:.1%}<br>' +
                  '<extra></extra>'
)

# Improve animation controls
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500

fig.show()


**Output:**

*[Interactive scatter plot: "Company Growth Animation: Revenue vs Profitability (2020-2024)", x: revenue, y: profit_margin]*

**Animation Parameters:**

**animation_frame**: Variable that defines time steps

- Each unique value becomes a frame
- Example: `'year'` creates 5 frames (2020-2024)

**animation_group**: Variable that tracks objects across frames

- Keeps same company as same point through animation
- Enables smooth transitions

**Fixed Ranges (Important!):**


In [ ]:
range_x=[0, 500],
range_y=[0, 0.35]


Without fixed ranges, axes rescale each frame (disorienting).

**Animation Speed:**


In [ ]:
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000  # 1 second per frame
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 500  # 0.5s transition


**When to Use Animation:**

- Presenting to audience (engaging)
- Showing temporal progression (growth, decline)
- Demonstrating cause-effect with timing
- Limited when: User needs to compare non-sequential frames

**3D Visualizations - Adding Another Dimension**

**Three-Dimensional Data:**

Some business problems naturally have three continuous dimensions:

- Geographic data (latitude, longitude, elevation/value)
- Product data (price, quality, market share)
- Customer segments (age, income, spending)

**Creating 3D Scatter Plot:**


In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

# Generate customer data with 3 dimensions
np.random.seed(42)
n_customers = 300

df_customers = pd.DataFrame({
    'customer_id': range(1, n_customers + 1),
    'age': np.random.normal(40, 15, n_customers),
    'annual_income': np.random.normal(65000, 25000, n_customers),
    'annual_spending': np.random.normal(25000, 10000, n_customers),
    'segment': np.random.choice(['Budget', 'Standard', 'Premium'], n_customers)
})

# Ensure reasonable values
df_customers['age'] = df_customers['age'].clip(18, 80)
df_customers['annual_income'] = df_customers['annual_income'].clip(20000, 200000)
df_customers['annual_spending'] = df_customers['annual_spending'].clip(5000, 80000)

# Create realistic relationships
df_customers['annual_spending'] = (
    5000 + 
    0.2 * df_customers['annual_income'] +
    np.random.normal(0, 5000, n_customers)
)

# Segment based on behavior
df_customers['segment'] = pd.cut(
    df_customers['annual_spending'] / df_customers['annual_income'],
    bins=[0, 0.2, 0.4, 1.0],
    labels=['Budget', 'Standard', 'Premium']
)

# Create 3D scatter plot
fig = px.scatter_3d(df_customers,
                    x='age',
                    y='annual_income',
                    z='annual_spending',
                    color='segment',
                    size_max=10,
                    title='Customer Segmentation Analysis (3D)',
                    labels={
                        'age': 'Age (years)',
                        'annual_income': 'Annual Income ($)',
                        'annual_spending': 'Annual Spending ($)'
                    },
                    color_discrete_map={
                        'Budget': 'lightblue',
                        'Standard': 'lightgreen',
                        'Premium': 'gold'
                    })

# Format hover
fig.update_traces(
    hovertemplate='<b>Customer</b><br>' +
                  'Age: %{x:.0f}<br>' +
                  'Income: $%{y:,.0f}<br>' +
                  'Spending: $%{z:,.0f}<br>' +
                  '<extra></extra>'
)

# Update layout for better 3D viewing
fig.update_layout(
    scene=dict(
        xaxis_title='Age',
        yaxis_title='Income ($)',
        zaxis_title='Spending ($)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.3)  # Viewing angle
        )
    ),
    height=700
)

fig.show()


**Output:**

*[Interactive 3D scatter plot: "Customer Segmentation Analysis (3D)", x: age, y: annual_income]*

**3D Interactivity:**

Users can:

- **Rotate**: Click and drag to rotate the 3D space
- **Zoom**: Scroll to zoom in/out
- **Pan**: Shift + drag to pan
- **Hover**: See values for all 3 dimensions

**When to Use 3D:**

**Good use cases:**

- Geographic data with a third metric (map with elevation)
- Truly three-dimensional relationships
- Impressive presentations (when appropriate)

**Avoid 3D when:**

- 2D would work (3D adds complexity)
- Precise reading of values needed (2D clearer)
- Audience isn't comfortable with 3D visualization
- Printing (3D doesn't work on paper)

**Alternative: Multiple 2D views often better than 3D**

**Synchronized Subplots - Coordinated Multiple Views**

**The Coordinated View Concept:**

Show multiple related views that interact together:

- Hover on one plot highlights related data in others
- Zoom on one plot adjusts others
- Selection in one filters others

**Creating Synchronized Subplots:**


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Generate product data
np.random.seed(42)
products = ['Product A', 'Product B', 'Product C', 'Product D', 'Product E']
metrics = []

for product in products:
    metrics.append({
        'product': product,
        'revenue': np.random.uniform(500000, 2000000),
        'profit': np.random.uniform(50000, 400000),
        'units_sold': np.random.randint(5000, 25000),
        'customer_satisfaction': np.random.uniform(3.5, 4.8),
        'return_rate': np.random.uniform(0.02, 0.12)
    })

df_products = pd.DataFrame(metrics)

# Create subplots with shared x-axis
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=('Revenue by Product', 'Customer Satisfaction', 'Return Rate')
)

# Add revenue bars
fig.add_trace(
    go.Bar(x=df_products['product'], y=df_products['revenue'],
           name='Revenue', marker_color='lightblue',
           hovertemplate='%{x}<br>Revenue: $%{y:,.0f}<extra></extra>'),
    row=1, col=1
)

# Add satisfaction line
fig.add_trace(
    go.Scatter(x=df_products['product'], y=df_products['customer_satisfaction'],
               mode='lines+markers', name='Satisfaction',
               line=dict(color='green', width=3),
               hovertemplate='%{x}<br>Satisfaction: %{y:.2f}/5<extra></extra>'),
    row=2, col=1
)

# Add return rate bars
fig.add_trace(
    go.Bar(x=df_products['product'], y=df_products['return_rate'],
           name='Return Rate', marker_color='salmon',
           hovertemplate='%{x}<br>Returns: %{y:.1%}<extra></extra>'),
    row=3, col=1
)

# Update layout
fig.update_layout(
    title_text='Product Performance Dashboard - Synchronized Views',
    showlegend=True,
    height=900,
    hovermode='x unified'  # Synchronize hover across all subplots
)

# Update axes
fig.update_yaxes(title_text='Revenue ($)', row=1, col=1, tickformat='$,.0f')
fig.update_yaxes(title_text='Satisfaction (1-5)', row=2, col=1)
fig.update_yaxes(title_text='Return Rate', row=3, col=1, tickformat='.1%')
fig.update_xaxes(title_text='Product', row=3, col=1)

fig.show()


**Output:**

*[Interactive Plotly chart]*

**Key Synchronization Features:**

**shared_xaxes=True:**

- All subplots share same X-axis
- Zoom on one subplot zooms all
- Maintains alignment

**hovermode='x unified':**

- Hover on one subplot shows data from all
- Single vertical line across all plots
- Compare metrics simultaneously

**Business Value:**

Hover over "Product C":

- See its revenue (top plot)
- See its satisfaction (middle plot)
- See its return rate (bottom plot)
- All at once - immediate insight into product health

**Custom Styling and Themes**

**Beyond Default Aesthetics:**

Professional applications require brand consistency:

- Company colors
- Corporate fonts
- Style guidelines
- Print/web variations

**Creating a Custom Theme:**


In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# Define company theme
COMPANY_COLORS = {
    'primary': '#1F4788',    # Corporate blue
    'secondary': '#F39C12',  # Accent orange
    'success': '#27AE60',    # Green
    'danger': '#E74C3C',     # Red
    'background': '#F8F9FA', # Light gray
    'text': '#2C3E50'        # Dark blue-gray
}

COMPANY_TEMPLATE = go.layout.Template(
    layout=go.Layout(
        font=dict(
            family='Arial, sans-serif',
            size=12,
            color=COMPANY_COLORS['text']
        ),
        title=dict(
            font=dict(size=18, color=COMPANY_COLORS['text']),
            x=0.5,
            xanchor='center'
        ),
        plot_bgcolor=COMPANY_COLORS['background'],
        paper_bgcolor='white',
        colorway=[COMPANY_COLORS['primary'], COMPANY_COLORS['secondary'],
                 COMPANY_COLORS['success'], COMPANY_COLORS['danger']],
        xaxis=dict(
            showgrid=True,
            gridcolor='white',
            showline=True,
            linecolor=COMPANY_COLORS['text'],
            linewidth=2
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='white',
            showline=True,
            linecolor=COMPANY_COLORS['text'],
            linewidth=2
        )
    )
)

# Use the custom template
df = pd.DataFrame({
    'category': ['Electronics', 'Accessories', 'Software', 'Services'],
    'sales': [45000, 28000, 35000, 22000]
})
fig = px.bar(df, x='category', y='sales')
fig.update_layout(template=COMPANY_TEMPLATE)
fig.show()


**Output:**

*[Interactive bar chart: x: category, y: Arial, sans-serif]*

**Applying Theme Globally:**


In [ ]:
import plotly.io as pio

# Set as default template
pio.templates['company_theme'] = COMPANY_TEMPLATE
pio.templates.default = 'company_theme'

# Now all plots use this theme automatically
df = pd.DataFrame({'x': range(5), 'y': [1, 4, 2, 5, 3]})
fig = px.scatter(df, x='x', y='y')
fig.show()  # Automatically uses company theme


**Output:**

*[Interactive scatter plot: x: x, y: y]*

### Lab Session
**Lab 2: Advanced Interactive Visualizations**

**Objective:** Master advanced Plotly techniques including multi-trace figures, animations, user controls, and 3D visualizations to create sophisticated, business-ready interactive dashboards.

**Scenario:** You're the Lead Data Visualization Engineer at "GlobalInsights Analytics," working for a Fortune 500 client launching a new product line. The executive committee needs advanced interactive visualizations for their quarterly board presentation. Standard charts won't suffice - they need visualizations that tell stories, respond to questions, and impress stakeholders. Your mission is to create cutting-edge interactive visualizations that showcase data sophistication.

**Pre-Lab Setup:**

1. Create file: `M4L02_YourName_Advanced.py`
2. Import libraries:


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np


3. Create output folder: "Lab2_Outputs"

**Part A: Multi-Metric Performance Dashboard with Dropdown Controls (30 points)**

**Context:** The CFO needs one dashboard that can switch between viewing Revenue, Profit, Operating Costs, and ROI - all with proper formatting and context for each metric.

**Data:**


In [ ]:
np.random.seed(42)

months = pd.date_range('2024-01-01', '2024-12-31', freq='M')

df_financial = pd.DataFrame({
    'month': months,
    'revenue': np.random.uniform(800000, 1200000, 12) + np.arange(12) * 10000,
    'costs': np.random.uniform(500000, 700000, 12) + np.arange(12) * 5000,
    'marketing': np.random.uniform(50000, 100000, 12),
    'r_and_d': np.random.uniform(80000, 150000, 12)
})

df_financial['profit'] = df_financial['revenue'] - df_financial['costs']
df_financial['operating_costs'] = df_financial['marketing'] + df_financial['r_and_d']
df_financial['roi'] = (df_financial['profit'] / df_financial['operating_costs']) * 100


**Tasks:**

1. **Create multi-trace figure with dropdown selector (20 points):**

   - Build using Graph Objects (not Express - need fine control)
   - Create 4 separate traces (Revenue, Profit, Operating Costs, ROI)
   - Initially show only Revenue (others hidden)
   - Add dropdown menu with 4 options to switch between metrics
   - Each option should:
     * Update trace visibility
     * Update Y-axis title
     * Update Y-axis format (currency for $, percentage for ROI)
     * Update colors appropriately (green for profit, red for costs, etc.)
   - Title: "Financial Performance Dashboard - Select Metric"

2. **Add target lines and annotations (5 points):**

   - Revenue target: $1,000,000 (horizontal dashed line)
   - Profit target: $400,000
   - ROI target: 150%
   - Target lines should update visibility with dropdown
   - Add annotation showing YTD average for displayed metric

3. **Professional styling (5 points):**

   - Dropdown positioned top-left, clearly labeled
   - Hover mode: 'x unified'
   - Grid lines for readability
   - Legend showing what's displayed
   - Height: 700 pixels
   - Save as: `M4L02_YourName_FinancialDashboard.html`

**Part B: Animated Market Share Evolution (25 points)**

**Context:** Marketing needs to show how market share evolved quarter-over-quarter for top 8 competitors, with animation revealing the competitive dynamics.

**Data:**


In [ ]:
np.random.seed(42)

quarters = ['Q1 2023', 'Q2 2023', 'Q3 2023', 'Q4 2023', 
            'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024']
companies = ['Our Company', 'Competitor A', 'Competitor B', 'Competitor C',
             'Competitor D', 'Competitor E', 'Competitor F', 'Others']

market_data = []
for i, quarter in enumerate(quarters):
    # Each company's market share evolves
    shares = {
        'Our Company': 22 + i * 1.5 + np.random.uniform(-1, 1),
        'Competitor A': 18 - i * 0.3 + np.random.uniform(-1, 1),
        'Competitor B': 15 + i * 0.5 + np.random.uniform(-1, 1),
        'Competitor C': 12 - i * 0.2 + np.random.uniform(-1, 1),
        'Competitor D': 11 + i * 0.3 + np.random.uniform(-1, 1),
        'Competitor E': 9 - i * 0.4 + np.random.uniform(-1, 1),
        'Competitor F': 7 + i * 0.1 + np.random.uniform(-1, 1),
    }
    
    # Others is remainder
    shares['Others'] = 100 - sum(shares.values())
    
    for company, share in shares.items():
        market_data.append({
            'quarter': quarter,
            'company': company,
            'market_share': max(share, 1)  # Ensure positive
        })

df_market = pd.DataFrame(market_data)


**Tasks:**

1. **Create animated bar chart race (15 points):**

   - Use animated horizontal bar chart
   - Animation frame: quarter
   - X-axis: market_share
   - Y-axis: company (sorted by share each frame)
   - Color by company (consistent colors throughout)
   - Fixed X-axis range: [0, 35]
   - Title: "Market Share Evolution: Quarterly Competitive Analysis"
   - Make "Our Company" stand out (different color, bold in legend)

2. **Optimize animation quality (5 points):**

   - Smooth transitions (500ms transition time)
   - Appropriate frame duration (1500ms)
   - Show current quarter prominently
   - Add annotations showing:
     * "Our Company" growth rate
     * Total market leader
   - Format as percentages with 1 decimal

3. **Business insights overlay (5 points):**

   - Add text box showing key insights:
     * Our Company's share in first vs last quarter
     * Biggest competitor threat
     * Market consolidation trend
   - Save as: `M4L02_YourName_MarketShareAnimation.html`
   - In comments, document the competitive story the animation tells

**Part C: 3D Product Portfolio Analysis (25 points)**

**Context:** The product team needs to visualize their portfolio across three dimensions: Price, Quality Score, and Market Demand, with segments identified.

**Data:**


In [ ]:
np.random.seed(42)

n_products = 50

df_portfolio = pd.DataFrame({
    'product_id': [f'SKU-{i:03d}' for i in range(1, n_products + 1)],
    'price': np.random.uniform(20, 500, n_products),
    'quality_score': np.random.uniform(60, 100, n_products),
    'market_demand': np.random.uniform(1000, 50000, n_products),
    'profit_margin': np.random.uniform(0.10, 0.40, n_products),
    'category': np.random.choice(['Electronics', 'Accessories', 'Software'], n_products)
})

# Create natural relationships
df_portfolio['market_demand'] = df_portfolio['market_demand'] * (
    1 + (100 - df_portfolio['price']) / 100
)  # Lower price = higher demand

df_portfolio['profit_margin'] = df_portfolio['profit_margin'] * (
    1 + (df_portfolio['quality_score'] - 60) / 100
)  # Higher quality = higher margin

# Strategic segments
df_portfolio['strategic_segment'] = 'Standard'
df_portfolio.loc[(df_portfolio['quality_score'] > 85) & 
                 (df_portfolio['profit_margin'] > 0.25), 'strategic_segment'] = 'Premium Stars'
df_portfolio.loc[(df_portfolio['market_demand'] > 30000) & 
                 (df_portfolio['profit_margin'] > 0.20), 'strategic_segment'] = 'Volume Winners'
df_portfolio.loc[(df_portfolio['quality_score'] < 70) | 
                 (df_portfolio['profit_margin'] < 0.15), 'strategic_segment'] = 'Review Needed'


**Tasks:**

1. **Create interactive 3D scatter plot (15 points):**

   - X-axis: price
   - Y-axis: quality_score
   - Z-axis: market_demand
   - Color by strategic_segment
   - Size by profit_margin
   - Include product_id and category in hover
   - Title: "Product Portfolio Analysis: Price-Quality-Demand Matrix (3D)"
   - Custom colors:
     * Premium Stars: gold
     * Volume Winners: green
     * Standard: lightblue
     * Review Needed: red

2. **Enhance 3D visualization (5 points):**

   - Set optimal camera angle for presentation
   - Add axis labels with units
   - Format hover to show:
     * Product ID (bold)
     * Category
     * Price: "$XXX.XX"
     * Quality: "XX/100"
     * Demand: "XX,XXX units"
     * Profit Margin: "XX.X%"
   - Size range: 5-20 (readable but not overwhelming)

3. **Strategic insights (5 points):**

   - In code comments, identify:
     * How many "Premium Stars" products?
     * Characteristics of "Volume Winners"
     * Which products need review and why?
     * Recommendation: Where should R&D focus?
   - Create 2D projection (Price vs Quality colored by segment) for comparison
   - Save both: `M4L02_YourName_Portfolio3D.html` and `M4L02_YourName_Portfolio2D.html`

**Part D: Synchronized Multi-View Sales Dashboard (20 points)**

**Context:** Regional managers need a dashboard showing sales performance, with synchronized views of different metrics that highlight together.

**Data:**


In [ ]:
np.random.seed(42)

regions = ['North', 'South', 'East', 'West', 'Central']

df_regional = pd.DataFrame({
    'region': regions,
    'q1_sales': [450000, 380000, 520000, 410000, 390000],
    'q2_sales': [470000, 390000, 550000, 430000, 405000],
    'q3_sales': [490000, 405000, 570000, 445000, 420000],
    'q4_sales': [580000, 480000, 650000, 520000, 495000],
    'customer_count': [2500, 2100, 2900, 2300, 2200],
    'avg_deal_size': [800, 750, 900, 850, 775],
    'satisfaction': [4.3, 4.1, 4.5, 4.2, 4.0]
})

df_regional['total_sales'] = (df_regional['q1_sales'] + df_regional['q2_sales'] + 
                               df_regional['q3_sales'] + df_regional['q4_sales'])
df_regional['growth_rate'] = (df_regional['q4_sales'] - df_regional['q1_sales']) / df_regional['q1_sales']


**Tasks:**

1. **Create synchronized 4-panel dashboard (15 points):**

   - Use `make_subplots()` with 2 rows, 2 columns
   - Panel 1 (top-left): Total sales by region (bar chart)
   - Panel 2 (top-right): Growth rate by region (horizontal bar)
   - Panel 3 (bottom-left): Customer count vs satisfaction (scatter)
   - Panel 4 (bottom-right): Quarterly trend (line chart, all regions)
   - Shared hover: hovering over a region in any panel highlights it everywhere
   - Title: "Regional Sales Performance - Synchronized Dashboard"

2. **Implement interactivity (3 points):**

   - Configure `hovermode='x unified'` where appropriate
   - Make panels share color coding (North always same color)
   - Enable legend click to filter
   - Format all currency as "$XXX,XXX"
   - Format percentages properly

3. **Layout optimization (2 points):**

   - Appropriate spacing between subplots
   - Clear subplot titles
   - Overall figure height: 900 pixels
   - All text readable
   - Save as: `M4L02_YourName_SynchronizedDashboard.html`

**Bonus Challenge (+25 points):**

**Create an Executive Decision Support Tool:**

Build an advanced interactive dashboard combining multiple techniques:

1. **Main view**: Animated scatter plot showing company performance over time
   - X: Revenue, Y: Profit Margin, Size: Employees, Color: Division
   - Animation: Quarterly progression

2. **Control panel**: Buttons and dropdowns for:
   - Metric selection (Revenue/Profit/ROI/Customer Sat)
   - Time period (Quarterly/Monthly/YTD)
   - Division filter (All/Sales/Engineering/Operations)
   - Chart type toggle (Scatter/Line/Bar)

3. **Side panel**: Synchronized KPI cards showing:
   - Selected metric's current value
   - Change from previous period
   - Rank among divisions
   - Projected next quarter (simple trend)

4. **Interactive features**:
   - Click a division to isolate it
   - Hover shows detailed drill-down
   - Export selected view as PNG
   - Reset button to default view

**Requirements:**

- Use Graph Objects for complete control
- Professional corporate styling
- Smooth animations
- Responsive layout
- Comprehensive hover information
- Save as: `M4L02_YourName_ExecutiveTool.html`

**Deliverables:**

1. Python file: `M4L02_YourName_Advanced.py`
2. HTML files (5-6):
   - `M4L02_YourName_FinancialDashboard.html`
   - `M4L02_YourName_MarketShareAnimation.html`
   - `M4L02_YourName_Portfolio3D.html`
   - `M4L02_YourName_Portfolio2D.html`
   - `M4L02_YourName_SynchronizedDashboard.html`
   - `M4L02_YourName_ExecutiveTool.html` (bonus)

**Grading Rubric:**

- Part A (Dropdown Dashboard): 30 points
- Part B (Market Share Animation): 25 points
- Part C (3D Portfolio): 25 points
- Part D (Synchronized Dashboard): 20 points
- Bonus (Executive Tool): +25 points

**Success Criteria:**

- [ ] Dropdown menus work smoothly
- [ ] Animations play without glitches
- [ ] 3D plot is interactive (rotate, zoom)
- [ ] Synchronized hover works across subplots
- [ ] All formatting (currency, %) correct
- [ ] Professional appearance
- [ ] Business insights documented
- [ ] HTML files open and function in browser

**Advanced Techniques Checklist:**

- [ ] updatemenus configured correctly
- [ ] Animation frame and group parameters set
- [ ] 3D camera positioned for optimal viewing
- [ ] Subplots synchronized with shared axes/hover
- [ ] Custom color schemes applied
- [ ] Hover templates formatted professionally
- [ ] Appropriate chart types for each data story
- [ ] Interactivity tested (clicks, hovers, zooms)

**Common Mistakes to Avoid:**

- Dropdown buttons that don't update properly
- Animation with unfixed axis ranges (disorienting)
- 3D plots with default camera angle (poor view)
- Unsynchronized subplots (defeating the purpose)
- Inconsistent color schemes across views
- Missing hover information
- Slow-loading animations (too many data points)
